# 02 — Naive Straw Man: Why Before/After Lies

Goal: Produce visually compelling evidence that naive analysis gives the wrong answer.

Hook: 'This naive view says the change made things worse. It didn't. Here's what the naive view is missing.'

Key plots for the Quarto report:
- Failure rates by install cohort (seasonal dip makes new design look better for wrong reasons)
- Treatment probability vs. install date (shows selection bias)
- Simpson's paradox table: within region, B is better; overall, naive comparison misleads

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/field_panel.csv', parse_dates=['install_date'])
df['install_year_month'] = df.install_date.dt.to_period('M').astype(str)

%matplotlib inline
sns.set_theme(style='darkgrid')

In [ ]:
# The three confounders visualized
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Selection bias: install date → design variant
df['install_month_num'] = df.install_date.dt.month + (df.install_date.dt.year - 2022) * 12
trt = df.groupby('install_month_num').apply(lambda x: (x.design_variant == 'B').mean())
axes[0].plot(trt.index, trt.values)
axes[0].set_title('Selection Bias: P(Variant B) by Install Month')
axes[0].set_xlabel('Months since Jan 2022')
axes[0].set_ylabel('Fraction receiving B')

# 2. Seasonality: failure rate by install month
seasonal = df.groupby('install_month_num').failure_event.mean() * 100
axes[1].plot(seasonal.index, seasonal.values, color='tomato')
axes[1].set_title('Seasonality Confound: Failure Rate by Install Month')
axes[1].set_xlabel('Months since Jan 2022')
axes[1].set_ylabel('Failure rate (%)')

# 3. Regional confound
regional = df.groupby(['region', 'design_variant']).failure_event.mean().unstack() * 100
regional.plot(kind='bar', ax=axes[2], color=['tomato', 'steelblue'])
axes[2].set_title('Regional Confound: Failure Rate by Region + Variant')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Three Confounders That Bias the Naive Estimate', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/naive_confounders.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Simpson's paradox: within each region, variant B is better
# But naive overall comparison can be misleading
within_region = df.groupby(['region', 'design_variant']).failure_event.mean().unstack() * 100
within_region['B_minus_A'] = within_region['B'] - within_region['A']
within_region['direction_correct'] = within_region['B_minus_A'] < 0
print('Within-region comparison (B vs. A):')
print(within_region.round(2))
print('\nAll regions should show B < A — the regional analysis is correct')
print('Naive pooled comparison is misleading because of unequal region mix in treatment groups')